In [1]:
!pip install -q transformers datasets torch scikit-learn pandas matplotlib seaborn accelerate sentencepiece

from google.colab import drive
drive.mount("/content/drive")

import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
)

# =========================
# CONFIG
# =========================
DATA_PATH = "/content/drive/MyDrive/conversation_cn.json"
RESULTS_DIR = "/content/drive/MyDrive/chinese/thesis_final_results_5cv_original_bert"
os.makedirs(RESULTS_DIR, exist_ok=True)

MODEL_FOR_CV = "bert-base-chinese"
MAX_LEN = 256
RANDOM_STATE = 42
VAL_SIZE_WITHIN_TRAIN = 0.20
N_FOLDS = 5

VALID_LABELS = ["Chitchat", "Interpret", "Reveal", "Revise"]
label2id = {"Chitchat": 0, "Interpret": 1, "Reveal": 2, "Revise": 3}
id2label = {v: k for k, v in label2id.items()}

# =========================
# LOAD ORIGINAL CHINESE DATA
# =========================
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

all_rows = []
for conv in raw:
    conv_id = str(conv.get("conv_id"))
    for i, t in enumerate(conv.get("conversation", [])):
        all_rows.append({
            "conversation_id": conv_id,
            "turn_index": int(i),
            "role": str(t.get("role", "")).lower(),
            "text": t.get("content") or "",
            "intent": t.get("intent"),
        })

df_turns_full = pd.DataFrame(all_rows).sort_values(
    ["conversation_id", "turn_index"]
).reset_index(drop=True)

df_turns_full["conversation_id"] = df_turns_full["conversation_id"].astype(str)
df_turns_full["turn_index"] = df_turns_full["turn_index"].astype(int)
df_turns_full["role"] = df_turns_full["role"].astype(str).str.lower()
df_turns_full["text"] = df_turns_full["text"].fillna("").astype(str)

target_df = df_turns_full[
    (df_turns_full["role"] == "user") &
    (df_turns_full["intent"].notna()) &
    (df_turns_full["intent"].isin(VALID_LABELS))
].copy().reset_index(drop=True)

turns_by_conv_full = {
    cid: g.sort_values("turn_index").reset_index(drop=True)
    for cid, g in df_turns_full.groupby("conversation_id")
}

print("Target rows:", len(target_df))
print("Conversations:", target_df["conversation_id"].nunique())
print(target_df["intent"].value_counts())

# =========================
# HELPERS
# =========================
def build_context_dataset(df_split, turns_by_conv):
    X, y = [], []

    for _, row in df_split.iterrows():
        cid = str(row["conversation_id"])
        ti = int(row["turn_index"])

        conv_turns = turns_by_conv[cid]
        prev_turns = conv_turns[conv_turns["turn_index"] < ti]

        history = []
        for _, p in prev_turns.iterrows():
            role = str(p["role"]).upper().strip()
            text = str(p["text"]).strip()
            if text:
                history.append(f"{role}: {text}")

        current_turn = str(row["text"]).strip()

        if history:
            full_text = "\n".join(history) + "\n" + f"USER: {current_turn}"
        else:
            full_text = f"USER: {current_turn}"

        X.append(full_text)
        y.append(row["intent"])

    return np.array(X), np.array(y)


class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        weight = self.class_weights.to(device=logits.device, dtype=logits.dtype)
        loss_fct = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1),
        )

        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    p, r, macro_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    weighted_f1 = f1_score(
        labels, preds, average="weighted", zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "macro_precision": p,
        "macro_recall": r,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }


def make_hf_datasets(tokenizer, X_tr, y_tr_ids, X_val, y_val_ids, X_te, y_te_ids):
    tr_enc = tokenizer(list(X_tr), truncation=True, padding=True, max_length=MAX_LEN)
    val_enc = tokenizer(list(X_val), truncation=True, padding=True, max_length=MAX_LEN)
    te_enc = tokenizer(list(X_te), truncation=True, padding=True, max_length=MAX_LEN)

    train_ds = Dataset.from_dict({
        "input_ids": tr_enc["input_ids"],
        "attention_mask": tr_enc["attention_mask"],
        "labels": list(y_tr_ids),
    })

    val_ds = Dataset.from_dict({
        "input_ids": val_enc["input_ids"],
        "attention_mask": val_enc["attention_mask"],
        "labels": list(y_val_ids),
    })

    test_ds = Dataset.from_dict({
        "input_ids": te_enc["input_ids"],
        "attention_mask": te_enc["attention_mask"],
        "labels": list(y_te_ids),
    })

    return train_ds, val_ds, test_ds


def evaluate_and_report(trainer, dataset, y_ids, X_texts, split_name, model_name, fold):
    pred_out = trainer.predict(dataset)
    y_pred_ids = np.argmax(pred_out.predictions, axis=1)

    y_pred = [id2label[i] for i in y_pred_ids]
    y_true = [id2label[i] for i in y_ids]

    report = classification_report(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        digits=3,
        zero_division=0,
    )

    print(f"\n=== {model_name} - {split_name} FOLD {fold} REPORT ===")
    print(report)

    cm = confusion_matrix(y_true, y_pred, labels=VALID_LABELS)
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{l}" for l in VALID_LABELS],
        columns=[f"pred_{l}" for l in VALID_LABELS],
    )

    tag = f"{model_name.lower().replace('/', '_')}_fold_{fold}_{split_name.lower()}"

    pred_df = pd.DataFrame({
        "text": X_texts,
        "true_label": y_true,
        "predicted_label": y_pred,
    })

    pred_df.to_csv(os.path.join(RESULTS_DIR, f"{tag}_predictions.csv"), index=False)

    errors = pred_df[pred_df["true_label"] != pred_df["predicted_label"]]

    confused_pairs = (
        errors[["true_label", "predicted_label"]]
        .value_counts()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    confused_pairs.to_csv(
        os.path.join(RESULTS_DIR, f"{tag}_confused_pairs.csv"),
        index=False,
    )

    with open(os.path.join(RESULTS_DIR, f"{tag}_report.txt"), "w", encoding="utf-8") as f:
        f.write(report)
        f.write("\n\n=== CONFUSION MATRIX ===\n")
        f.write(cm_df.to_string())

    metrics = compute_metrics((pred_out.predictions, np.array(y_ids)))

    _, _, per_class_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        average=None,
        zero_division=0,
    )

    for label, f1_val in zip(VALID_LABELS, per_class_f1):
        metrics[f"{label.lower()}_f1"] = f1_val

    return metrics


def run_model_for_fold(
    model_name,
    fold,
    X_train,
    y_train_ids,
    X_val,
    y_val_ids,
    X_test,
    y_test_ids,
    class_weights_tensor,
    num_epochs=5,
):
    print("\n" + "=" * 70)
    print(f"MODEL: {model_name} | FOLD {fold}")
    print("=" * 70)

    set_seed(RANDOM_STATE + fold)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_ds, val_ds, test_ds = make_hf_datasets(
        tokenizer,
        X_train,
        y_train_ids,
        X_val,
        y_val_ids,
        X_test,
        y_test_ids,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(VALID_LABELS),
        id2label=id2label,
        label2id=label2id,
    )

    short_name = model_name.split("/")[-1]

    training_args = TrainingArguments(
        output_dir=os.path.join(RESULTS_DIR, f"{short_name}_fold_{fold}_results"),
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.05,
        seed=RANDOM_STATE + fold,
        data_seed=RANDOM_STATE + fold,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    trainer = WeightedTrainer(
        class_weights=class_weights_tensor,
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    test_metrics = evaluate_and_report(
        trainer,
        test_ds,
        y_test_ids,
        X_test,
        "TEST",
        short_name,
        fold,
    )

    test_metrics["fold"] = fold
    test_metrics["model"] = short_name
    test_metrics["test_samples"] = len(y_test_ids)

    return test_metrics


# =========================
# 5-FOLD CV
# =========================
gkf = GroupKFold(n_splits=N_FOLDS)
groups = target_df["conversation_id"]

fold_results = []
fold_sizes = []

for fold, (train_val_idx, test_idx) in enumerate(
    gkf.split(target_df, groups=groups),
    start=1,
):
    print("\n" + "=" * 90)
    print(f"FOLD {fold}/{N_FOLDS}")
    print("=" * 90)

    train_val_df = target_df.iloc[train_val_idx].reset_index(drop=True)
    test_df = target_df.iloc[test_idx].reset_index(drop=True)

    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=VAL_SIZE_WITHIN_TRAIN,
        random_state=RANDOM_STATE + fold,
    )

    train_idx, val_idx = next(
        gss.split(train_val_df, groups=train_val_df["conversation_id"])
    )

    train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
    val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

    print("Train:", len(train_df))
    print("Val:", len(val_df))
    print("Test:", len(test_df))

    train_cids = set(train_df["conversation_id"].astype(str))
    val_cids = set(val_df["conversation_id"].astype(str))
    test_cids = set(test_df["conversation_id"].astype(str))

    print("Train ∩ Val:", len(train_cids & val_cids))
    print("Train ∩ Test:", len(train_cids & test_cids))
    print("Val ∩ Test:", len(val_cids & test_cids))

    if (train_cids & val_cids) or (train_cids & test_cids) or (val_cids & test_cids):
        raise ValueError(f"Conversation leakage detected in fold {fold}.")

    X_train, y_train = build_context_dataset(train_df, turns_by_conv_full)
    X_val, y_val = build_context_dataset(val_df, turns_by_conv_full)
    X_test, y_test = build_context_dataset(test_df, turns_by_conv_full)

    y_train_ids = np.array([label2id[l] for l in y_train])
    y_val_ids = np.array([label2id[l] for l in y_val])
    y_test_ids = np.array([label2id[l] for l in y_test])

    classes = np.array(sorted(np.unique(y_train_ids)))

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train_ids,
    )

    full_class_weights = np.ones(len(VALID_LABELS), dtype=np.float32)
    for cls_id, w in zip(classes, class_weights):
        full_class_weights[cls_id] = w

    class_weights_tensor = torch.tensor(full_class_weights, dtype=torch.float)

    print("Class weights:")
    for cls_id, w in enumerate(full_class_weights):
        print(f"  {id2label[cls_id]}: {w:.3f}")

    fold_metrics = run_model_for_fold(
        MODEL_FOR_CV,
        fold,
        X_train,
        y_train_ids,
        X_val,
        y_val_ids,
        X_test,
        y_test_ids,
        class_weights_tensor,
        num_epochs=5,
    )

    fold_results.append(fold_metrics)

    fold_sizes.append({
        "fold": fold,
        "train_rows": len(train_df),
        "val_rows": len(val_df),
        "test_rows": len(test_df),
    })

# =========================
# SAVE SUMMARY
# =========================
cv_results_df = pd.DataFrame(fold_results)
fold_sizes_df = pd.DataFrame(fold_sizes)

summary_cols = [
    "accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
    "chitchat_f1",
    "interpret_f1",
    "reveal_f1",
    "revise_f1",
]

summary = cv_results_df[summary_cols].agg(["mean", "std"])

print("\n===== 5-FOLD CV RESULTS =====")
display(cv_results_df)

print("\n===== 5-FOLD CV SUMMARY =====")
display(summary)

print("\n===== FOLD SIZES =====")
display(fold_sizes_df)

safe_model_name = MODEL_FOR_CV.replace("/", "_")

cv_results_df.to_csv(
    os.path.join(RESULTS_DIR, f"{safe_model_name}_5fold_cv_results.csv"),
    index=False,
)

summary.to_csv(
    os.path.join(RESULTS_DIR, f"{safe_model_name}_5fold_cv_summary.csv"),
)

fold_sizes_df.to_csv(
    os.path.join(RESULTS_DIR, f"{safe_model_name}_5fold_sizes.csv"),
    index=False,
)

print("Done. Saved results to:", RESULTS_DIR)

Mounted at /content/drive
Target rows: 3055
Conversations: 904
intent
Interpret    1590
Reveal        975
Chitchat      275
Revise        215
Name: count, dtype: int64

FOLD 1/5
Train: 1955
Val: 489
Test: 611
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0
Class weights:
  Chitchat: 2.842
  Interpret: 0.480
  Reveal: 0.777
  Revise: 3.620

MODEL: bert-base-chinese | FOLD 1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

...


===== 5-FOLD CV SUMMARY =====


,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,chitchat_f1,interpret_f1,reveal_f1,revise_f1
mean,0.937807,0.916735,0.895645,0.905123,0.937564,0.911789,0.948610,0.956454,0.803637
std,0.013296,0.021526,0.023780,0.021175,0.013131,0.020611,0.011654,0.014002,0.070381



===== FOLD SIZES =====


,fold,train_rows,val_rows,test_rows
0,1,1955,489,611
1,2,1949,495,611
2,3,1959,485,611
3,4,1948,496,611
4,5,1951,493,611


Done. Saved results to: /content/drive/MyDrive/chinese/thesis_final_results_5cv_original_bert
